In [1]:
!wget -q https://vovakim.com/projects/CorrsBlended/SurfCorr2.0.benchmarks.bins.zip
!unzip -q SurfCorr2.0.benchmarks.bins.zip
!mkdir -p /content/TOSCA/raw/
!cp -r CorrsBenchmark/Data/TOSCA/Meshes/* /content/TOSCA/raw/

!git clone -q https://github.com/tpitois/FLBO
%cd FLBO
!pip install -q torch_geometric libigl

/content/FLBO
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.1 MB/s eta 0:00:00:00:01:01


In [2]:
import torch
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm

from src.datasets.TOSCA import TOSCA
from src.datasets.transforms import FLBOTransform
from src.models.ascnn import ACSCNN
from src.utils.eval import evaluate_predictions

In [3]:
dataset = TOSCA(
    root='/content/TOSCA/',
    category='Cat',
    pre_transform=FLBOTransform(n_angles=8, alpha=10.0, tau=0.5)
)

train_dataset = dataset[:9]
test_dataset = dataset[9:]

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

Processing...


Lecture de cat0.off...
Lecture de cat1.off...
Lecture de cat2.off...
Lecture de cat3.off...
Lecture de cat4.off...
Lecture de cat5.off...
Lecture de cat6.off...
Lecture de cat7.off...
Lecture de cat8.off...
Lecture de cat9.off...
Lecture de cat10.off...
Application du pré-traitement (FLBO, WKS...)...
Dataset traité et sauvegardé avec succès !


Done!


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

n_desc = dataset[0].x.size(1)
n_class = dataset[0].num_nodes

model = ACSCNN(n_desc, n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
criterion = torch.nn.CrossEntropyLoss()

num_epochs = 100

scaler = torch.amp.GradScaler('cuda')

In [ ]:
pbar = tqdm(range(num_epochs), leave=True)

for epoch in pbar:
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_nodes = 0

    for i, data in enumerate(train_loader):
        x = data.x.to(device)
        L = data.L.to(device)
        y = data.y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(x, L)
            loss = criterion(outputs, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item()
        running_corrects += torch.sum(preds == y).item()
        total_nodes += y.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = running_corrects / total_nodes

    pbar.set_postfix({
    'loss': f'{epoch_loss:.4f}',
    'acc': f'{epoch_acc:.4f}'
    })

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
model.eval()

test_data = test_dataset[0]
x = test_data.x.to(device)
L = test_data.L.to(device)
y = test_data.y.cpu().numpy()

with torch.no_grad():
    with torch.amp.autocast('cuda'):
        outputs = model(x, L)
        _, preds = torch.max(outputs, 1)

preds = preds.cpu().numpy()

V = test_data.pos.numpy()
F = test_data.face.t().numpy()

errors = evaluate_predictions(preds, y, V, F, num_samples=1000)